# 12. Qwen2.5-VL Streamlit 멀티모달 챗 — CPU 16GB 로컬 실행

이 노트북은 **Ollama 로컬 양자화 모델 + Streamlit**을 사용한다. 모델은 최초 1회 다운로드가 필요하며, 이후 localhost에서 실행한다.

- 이미지 업로드, 미리보기, 한국어 질의응답, 멀티턴 대화, 이미지 비교 지원
- 기존 Transformers float32 3B 모델 로딩을 제거하여 메모리 부담을 줄임
- CPU에서도 느릴 수 있으므로 1장·336px·32토큰부터 테스트


# 1. Streamlit에서 중요한 점

Streamlit은 사용자가 Widget을 조작할 때 Python Script를 다시 실행한다.

따라서 일반 Python 변수만 사용하면 대화 History를 안정적으로 유지하기 어렵다.

이를 위해 다음 구조를 사용한다.

```text
st.session_state
├── messages
├── images
└── image_signature
```

`messages`에는 대화 History를 저장한다.

`images`에는 현재 대화에서 사용할 PIL Image를 저장한다.

`image_signature`는 업로드 이미지가 변경되었는지 판단하는 데 사용한다.

# 2. CPU 메모리와 모델 로딩

16GB RAM PC에서는 `Qwen2_5_VLForConditionalGeneration.from_pretrained(..., torch_dtype=torch.float32)`가 메모리 부족을 일으킬 수 있다.

이 버전은 **Ollama가 별도 프로세스에서 양자화 모델을 관리**하므로 Streamlit은 모델 가중치를 직접 로드하지 않는다.

`st.cache_resource`는 이 구현에서 필요하지 않다. 첫 응답은 모델 로딩으로 오래 걸릴 수 있다.


# 3. 프로젝트 구조

Notebook을 실행하면 같은 폴더에 다음 파일을 생성한다.

```text
qwen_vl_streamlit/
│
├── app.py
└── README.txt
```

실제 웹 애플리케이션은 `app.py`에 구현한다.

# 4. 프로젝트 폴더 생성

In [1]:
# ---------------------------------------------------------
# 운영체제에 독립적인 파일 경로 처리를 위해 Path를 불러온다.
# ---------------------------------------------------------
from pathlib import Path

# ---------------------------------------------------------
# Streamlit 프로젝트 폴더 경로를 지정한다.
# ---------------------------------------------------------
PROJECT_DIR = Path(
    "./qwen_vl_streamlit"
)

# ---------------------------------------------------------
# 프로젝트 폴더가 없으면 새로 생성한다.
# ---------------------------------------------------------
PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------------
# Streamlit 애플리케이션 파일 경로를 지정한다.
# ---------------------------------------------------------
APP_PATH = PROJECT_DIR / "app.py"

# ---------------------------------------------------------
# 실행 안내 파일 경로를 지정한다.
# ---------------------------------------------------------
README_PATH = PROJECT_DIR / "README.txt"

# ---------------------------------------------------------
# 프로젝트 폴더의 절대 경로를 출력한다.
# ---------------------------------------------------------
print(
    "Project Directory:",
    PROJECT_DIR.resolve()
)

Project Directory: C:\sk-encoa\llm-workspace\멀티모달_CLIP_BLIP\qwen_vl_streamlit


# 5. Streamlit 애플리케이션 전체 소스

아래 문자열을 `app.py`로 저장한다. 모델 추론은 `http://127.0.0.1:11434/api/chat`의 로컬 Ollama 서버에 요청한다.


In [2]:
APP_SOURCE = '# Qwen2.5-VL 로컬 CPU 멀티모달 챗 (Ollama + Streamlit)\n# 설치: pip install streamlit requests pillow\n# Ollama 모델 준비: ollama pull qwen2.5vl:3b\n# 실행: python -m streamlit run app.py\nimport base64\nimport hashlib\nimport io\nimport time\nimport requests\nimport streamlit as st\nfrom PIL import Image, ImageOps, UnidentifiedImageError\n\nst.set_page_config(page_title="로컬 Qwen2.5-VL 멀티모달 챗", layout="wide")\nst.title("Qwen2.5-VL 한국어 멀티모달 챗 (CPU)")\nst.caption("Ollama 로컬 양자화 모델 사용 · 외부 API 불필요 · 최초 모델 다운로드 필요")\n\n# Ollama는 별도의 프로세스에서 모델을 로드하고 CPU 추론을 수행한다.\nOLLAMA_URL = "http://127.0.0.1:11434/api/chat"\nMODEL_NAME = "qwen2.5vl:3b"\nMAX_IMAGES = 3\nMAX_UPLOAD_BYTES = 10 * 1024 * 1024\n\n# Session State에는 화면용 대화와 실제 모델에 보낼 메시지를 분리하여 저장한다.\ndef init_state():\n    defaults = {"messages": [], "image_signature": "", "images": [], "image_bytes": []}\n    for key, value in defaults.items():\n        if key not in st.session_state:\n            st.session_state[key] = value\n\ndef clear_history():\n    st.session_state.messages = []\n\ndef encode_image(uploaded_file, max_side):\n    # 이미지 형식 검사와 방향 보정을 수행한 뒤 RGB JPEG로 압축한다.\n    raw = uploaded_file.getvalue()\n    if len(raw) > MAX_UPLOAD_BYTES:\n        raise ValueError(f"{uploaded_file.name}: 파일 크기가 10MB를 초과합니다.")\n    try:\n        with Image.open(io.BytesIO(raw)) as opened:\n            image = ImageOps.exif_transpose(opened).convert("RGB")\n    except (UnidentifiedImageError, OSError) as exc:\n        raise ValueError(f"{uploaded_file.name}: 지원되지 않는 이미지입니다.") from exc\n    image.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)\n    output = io.BytesIO()\n    image.save(output, format="JPEG", quality=82, optimize=True)\n    return image, base64.b64encode(output.getvalue()).decode("ascii")\n\ndef create_signature(files, max_side):\n    # 이미지 크기 설정을 바꾸어도 새로 처리하도록 설정값을 포함한다.\n    digest = hashlib.sha256(str(max_side).encode("utf-8"))\n    for file in files:\n        digest.update(file.name.encode("utf-8"))\n        digest.update(file.getvalue())\n    return digest.hexdigest()\n\ndef check_ollama():\n    try:\n        response = requests.get("http://127.0.0.1:11434/api/tags", timeout=5)\n        response.raise_for_status()\n        models = [item.get("name", "") for item in response.json().get("models", [])]\n        available = any(name == MODEL_NAME or name.startswith(MODEL_NAME + "-") for name in models)\n        return True, available, models\n    except requests.RequestException:\n        return False, False, []\n\ndef build_messages(history, prompt, images_b64, max_turns):\n    # 이미지는 첫 질문에만 포함하고 후속 질문에는 텍스트 대화를 이어 붙인다.\n    # 최근 N개의 완성된 질문/답변 쌍만 유지해 CPU 처리량을 제한한다.\n    recent = history[-2 * max_turns:]\n    messages = [{"role": "system", "content": "이미지를 정확하게 관찰하고 한국어로 간결하게 답하세요. 확실하지 않으면 모른다고 답하세요."}]\n    first_prompt = next((m["content"] for m in history if m["role"] == "user"), prompt)\n    # 이미지 문맥을 잃지 않도록 최근 대화와 별도로 이미지 메시지를 항상 유지한다.\n    messages.append({"role": "user", "content": first_prompt, "images": images_b64})\n    if history:\n        # 원래 첫 질문을 중복 추가하지 않도록 제외한다.\n        rest = history[1:]\n        messages.extend(rest[-2 * max_turns:])\n        messages.append({"role": "user", "content": prompt})\n    return messages\n\ndef ask_ollama(messages, num_predict, timeout_seconds):\n    # 로컬 서버만 호출하며 외부 인터넷 API를 호출하지 않는다.\n    payload = {\n        "model": MODEL_NAME,\n        "messages": messages,\n        "stream": False,\n        "options": {"num_predict": num_predict, "temperature": 0},\n        "keep_alive": "10m",\n    }\n    started = time.perf_counter()\n    response = requests.post(OLLAMA_URL, json=payload, timeout=(10, timeout_seconds))\n    response.raise_for_status()\n    result = response.json()\n    answer = result.get("message", {}).get("content", "").strip()\n    return answer or "모델이 빈 응답을 반환했습니다.", time.perf_counter() - started\n\ninit_state()\nwith st.sidebar:\n    st.header("CPU 설정")\n    st.write("모델:", MODEL_NAME)\n    st.write("추론 장치: CPU (Ollama 설치/실행 설정 확인)")\n    max_side = st.select_slider("이미지 최대 변(px)", options=[224, 336, 448, 560, 672], value=336)\n    max_new_tokens = st.slider("최대 생성 토큰", 16, 128, 32, step=16)\n    max_turns = st.slider("유지할 최근 대화 쌍", 1, 5, 2)\n    timeout_seconds = st.slider("요청 대기 제한(초)", 60, 900, 300, step=60)\n    if st.button("대화 초기화"):\n        clear_history()\n        st.rerun()\n    if st.button("Ollama 연결 확인"):\n        connected, model_ready, installed = check_ollama()\n        if not connected:\n            st.error("Ollama 서버에 연결할 수 없습니다. Ollama 실행 여부를 확인하세요.")\n        elif not model_ready:\n            st.warning(f"모델이 없습니다. 터미널에서 ollama pull {MODEL_NAME} 실행")\n            st.write("설치 모델:", installed)\n        else:\n            st.success("Ollama와 모델을 확인했습니다.")\n\nst.subheader("이미지 업로드")\nuploaded_files = st.file_uploader("PNG/JPG/WEBP 이미지 (최대 3장, 각 10MB)", type=["png", "jpg", "jpeg", "webp"], accept_multiple_files=True)\nfiles = uploaded_files or []\nif len(files) > MAX_IMAGES:\n    st.warning(f"CPU 실습에서는 최대 {MAX_IMAGES}장만 사용합니다. 처음 {MAX_IMAGES}장을 선택합니다.")\n    files = files[:MAX_IMAGES]\nsignature = create_signature(files, max_side)\nif signature != st.session_state.image_signature:\n    try:\n        images, image_bytes = [], []\n        for file in files:\n            image, encoded = encode_image(file, max_side)\n            images.append(image)\n            image_bytes.append(encoded)\n        st.session_state.images = images\n        st.session_state.image_bytes = image_bytes\n        st.session_state.image_signature = signature\n        clear_history()\n    except ValueError as exc:\n        st.error(str(exc))\n        st.stop()\n\nif st.session_state.images:\n    cols = st.columns(min(3, len(st.session_state.images)))\n    for idx, img in enumerate(st.session_state.images):\n        with cols[idx % len(cols)]:\n            st.image(img, caption=f"Picture {idx + 1}: {img.width}×{img.height}", width="stretch")\nelse:\n    st.info("먼저 이미지를 업로드하세요.")\n\nfor message in st.session_state.messages:\n    with st.chat_message(message["role"]):\n        st.markdown(message["content"])\n\nprompt = st.chat_input("이미지에 대해 한국어로 질문하세요.")\nif prompt:\n    if not st.session_state.image_bytes:\n        st.warning("이미지를 먼저 업로드하세요.")\n    else:\n        with st.chat_message("user"):\n            st.markdown(prompt)\n        with st.chat_message("assistant"):\n            with st.spinner("로컬 CPU에서 추론 중입니다. 첫 요청은 모델 로딩으로 오래 걸릴 수 있습니다..."):\n                try:\n                    model_messages = build_messages(st.session_state.messages, prompt, st.session_state.image_bytes, max_turns)\n                    answer, elapsed = ask_ollama(model_messages, max_new_tokens, timeout_seconds)\n                    st.markdown(answer)\n                    st.caption(f"추론 시간: {elapsed:.1f}초 | 이미지: {len(st.session_state.images)}장 | 최대 생성: {max_new_tokens}토큰")\n                except requests.Timeout:\n                    st.error("응답 제한 시간을 초과했습니다. 이미지를 1장/224px, 출력 16토큰으로 줄여 재시도하세요.")\n                    st.stop()\n                except requests.RequestException as exc:\n                    st.error(f"Ollama 연결 또는 추론 오류: {exc}")\n                    st.info(f"Ollama 실행과 모델 설치를 확인하세요: ollama pull {MODEL_NAME}")\n                    st.stop()\n        st.session_state.messages.extend([{"role": "user", "content": prompt}, {"role": "assistant", "content": answer}])\n'
print("app.py source characters:", len(APP_SOURCE))


app.py source characters: 7225


# 6. `app.py` Python 문법 검사

파일로 저장하기 전에 `compile()`로 Python 문법을 검사한다.

In [3]:
# ---------------------------------------------------------
# Streamlit 전체 소스의 Python 문법을 검사한다.
# 오류가 있으면 이 셀에서 SyntaxError가 발생한다.
# ---------------------------------------------------------
compile(
    APP_SOURCE,
    "app.py",
    "exec"
)

# ---------------------------------------------------------
# 문법 검사가 완료되었음을 출력한다.
# ---------------------------------------------------------
print(
    "app.py syntax check: OK"
)

app.py syntax check: OK


# 7. `app.py` 파일 생성

In [4]:
# ---------------------------------------------------------
# Streamlit 전체 소스를 UTF-8 형식으로 app.py에 저장한다.
# ---------------------------------------------------------
APP_PATH.write_text(
    APP_SOURCE,
    encoding="utf-8"
)

# ---------------------------------------------------------
# 생성된 app.py 파일의 절대 경로를 출력한다.
# ---------------------------------------------------------
print(
    "Created:",
    APP_PATH.resolve()
)

Created: C:\sk-encoa\llm-workspace\멀티모달_CLIP_BLIP\qwen_vl_streamlit\app.py


# 8. 실행 안내 파일 작성

In [5]:
README_TEXT = 'Qwen2.5-VL Streamlit 멀티모달 챗 — Windows CPU 16GB RAM\n\n1. Ollama 설치: https://ollama.com/download/windows\n2. Windows PowerShell에서 CPU 전용으로 실행하려면 (Ollama 종료 후):\n   $env:OLLAMA_LLM_LIBRARY="cpu_avx2"\n   ollama serve\n   CPU에 AVX2가 없다면 cpu 또는 cpu_avx를 사용하세요. Ollama 버전에 따라 환경변수 지원이 다를 수 있습니다.\n3. 다른 PowerShell에서 모델 다운로드 (최초 1회 인터넷 필요):\n   ollama pull qwen2.5vl:3b\n4. Python 가상환경에서:\n   python -m pip install streamlit requests pillow\n   python -m streamlit run app.py\n5. 앱의 \'Ollama 연결 확인\' 버튼을 눌러 모델 준비 상태를 확인하세요.\n\n참고: CPU에서 3B 모델은 양자화해도 응답이 오래 걸릴 수 있습니다. 첫 실행은 모델 로딩으로 더 느립니다.\n권장 시작: 이미지 1장, 224~336px, 생성 토큰 16~32개.\n이 앱은 localhost Ollama API를 사용하며 외부 유료 API 호출은 없습니다.\n'
README_PATH.write_text(README_TEXT, encoding="utf-8")
print("README:", README_PATH.resolve())


README: C:\sk-encoa\llm-workspace\멀티모달_CLIP_BLIP\qwen_vl_streamlit\README.txt


# 9. 생성된 프로젝트 파일 확인

In [6]:
# ---------------------------------------------------------
# 프로젝트 폴더 안의 파일을 이름순으로 가져온다.
# ---------------------------------------------------------
project_files = sorted(
    PROJECT_DIR.iterdir()
)

# ---------------------------------------------------------
# 생성된 파일 이름을 하나씩 출력한다.
# ---------------------------------------------------------
for file_path in project_files:
    print(
        file_path.name
    )

app.py
README.txt


# 10. 애플리케이션 핵심 구조

```text
Browser → Streamlit → PIL 이미지 축소/JPEG Base64 → localhost Ollama /api/chat
                                                        ↓
                                               Qwen2.5-VL GGUF 양자화
                                                        ↓
                                                  CPU 한국어 답변
```

Transformers의 `processor.apply_chat_template()`과 `model.generate()`는 사용하지 않는다.


# 11. 왜 Session State가 필요한가

Streamlit은 Widget 입력이 발생할 때 Script를 다시 실행한다.

다음 일반 변수는 rerun 시 새로 생성된다.

```python
messages = []
```

반면 다음 상태는 사용자 Session 동안 유지할 수 있다.

```python
st.session_state.messages
```

따라서 대화형 UI에서는 Session State가 핵심이다.

# 12. UI History와 Model History를 분리한 이유

화면 표시용 History에는 문자열만 저장한다.

```python
{
    "role": "user",
    "content": "그 객체의 색은?"
}
```

반면 Qwen2.5-VL에 전달할 첫 User Message에는 실제 이미지가 필요하다.

```text
User
├── Image 1
├── Image 2
└── 첫 질문
```

따라서 `build_model_messages()`가 화면용 History와 현재 PIL Image를 결합하여 모델용 Multimodal History를 다시 만든다.

이렇게 하면 Session State 구조가 단순해지고 UI 출력과 모델 입력의 역할을 분리할 수 있다.

# 13. 첫 질문과 후속 질문의 차이

첫 질문에서는 이미지와 질문을 함께 전달한다.

```text
User
├── Picture 1
├── Picture 2
└── 두 이미지를 비교해줘.
```

두 번째 질문부터는 이전 History를 유지하면서 텍스트만 추가한다.

```text
Assistant
└── 첫 번째는 강아지이고 두 번째는 자동차입니다.

User
└── 첫 번째 이미지에 대해서만 더 설명해줘.
```

모델 입력에는 첫 Turn의 이미지가 여전히 포함되어 있으므로 후속 질문에서도 이미지 문맥을 참조할 수 있다.

# 14. 이미지가 변경되면 대화를 초기화하는 이유

이전 이미지를 대상으로 다음 대화를 진행했다고 가정한다.

```text
Image A
User: 이 동물은 무엇이야?
Assistant: 강아지입니다.
```

사용자가 새로운 Image B로 변경했는데 이전 History가 그대로 남아 있으면 `"이 동물"`이 어느 이미지를 의미하는지 혼동할 수 있다.

따라서 업로드 파일의 Hash를 계산하여 이미지가 변경되면 대화를 초기화한다.

```text
업로드 이미지 변경
      ↓
SHA-256 Signature 변경
      ↓
images 교체
      ↓
messages 초기화
```

# 15. 여러 이미지 비교 사용 예

이미지 2장을 업로드한 후 다음과 같이 질문할 수 있다.

```text
두 이미지의 주요 객체를 각각 알려줘.

첫 번째 이미지와 두 번째 이미지의 공통점은?

두 이미지의 가장 큰 차이는?

첫 번째 이미지에만 있는 특징을 알려줘.

두 번째 이미지에 대해서만 더 자세히 설명해줘.
```

`add_vision_id=True`를 사용하므로 여러 시각 입력을 참조하는 Prompt를 구성할 때 이미지 구분이 쉬워진다.

# 16. CPU 메모리 관리

- 모델 가중치는 Streamlit이 아닌 Ollama가 관리한다.
- 이미지 1장부터 시작하고, 최대 변 224~336px와 출력 16~32토큰을 권장한다.
- 여러 이미지를 업로드할수록 시각 토큰 처리 시간이 증가한다.
- `max_pixels`나 이미지 축소는 모델 가중치 크기를 줄이지 못한다.
- 모델 최초 다운로드에는 인터넷이 필요하지만 추론은 로컬에서 진행한다.


# 17. Windows CPU 실행 방법

1. Ollama Windows 설치: https://ollama.com/download/windows
2. PowerShell에서 `ollama pull qwen2.5vl:3b` 실행 (최초 1회)
3. Ollama가 실행 중인지 확인 (`ollama list`).
4. Ollama 실행이 안되여 있을때 (`ollama serve`).
5. `가상환경에서 uv add streamlit requests pillow`
6. `가상환경에서 streamlit run qwen_vl_streamlit/app.py`
7. 연결 확인 버튼 클릭 → 이미지 1장 → 짧은 한국어 질문.

GPU가 있는 PC에서 CPU만 강제하려면 Ollama의 GPU 사용 설정을 별도로 조정해야 한다.


# 18. 실습 시나리오 1 - 단일 이미지

이미지 한 장을 업로드한다.

질문 예:

```text
이 이미지의 주요 객체는 무엇이야?

그 객체의 색이나 외형을 설명해줘.

그 객체는 무엇을 하고 있어?

배경은 어떻게 보여?

지금까지 내용을 한 문장으로 정리해줘.
```

두 번째 질문부터는 이전 문맥을 유지하는지 확인한다.

# 19. 실습 시나리오 2 - 멀티이미지

이미지 두 장 이상을 업로드한다.

질문 예:

```text
각 이미지의 주요 객체를 알려줘.

Picture 1과 Picture 2의 공통점은?

두 이미지의 차이점을 설명해줘.

Picture 1에 대해서만 자세히 설명해줘.

Picture 2의 객체는 어떤 용도로 사용돼?
```

여러 이미지 참조가 정확하게 유지되는지 확인한다.

# 20. 실습 시나리오 3 - 오류 분석

다음 오류를 찾아 기록한다.

```text
Object Error
Attribute Error
Count Error
Image Confusion
Reference Resolution Error
Context Consistency Error
Hallucination
```

예:

```text
질문:
Picture 1의 객체는 무엇인가?

실제:
dog

모델:
cat

오류:
Object Error
```

또는

```text
질문:
두 번째 이미지의 색은?

모델:
첫 번째 이미지의 색을 설명함

오류:
Image Confusion
```

# 21. 확인 문제

1. Streamlit과 Ollama를 분리한 이유는 무엇인가?
2. float32 3B 모델이 16GB RAM 환경에서 위험한 이유는 무엇인가?
3. GGUF 양자화 모델이 메모리 사용량에 주는 영향은 무엇인가?
4. 이미지 해상도와 생성 토큰 수는 각각 어느 단계의 비용을 줄이는가?
5. `st.session_state`는 무엇을 유지하는가?
6. 이미지가 변경되면 대화 기록을 초기화하는 이유는 무엇인가?
7. `http://127.0.0.1:11434/api/chat`은 외부 API인가?


# 22. 확인 문제 정답

1. UI와 모델 메모리/추론 프로세스를 분리하기 위해서이다.
2. 모델 가중치만 약 12GB 수준이고 실행 중 추가 메모리가 필요하기 때문이다.
3. 가중치 표현 비트 수를 줄여 메모리 요구량을 낮춘다.
4. 이미지 크기는 시각 입력 처리량, 출력 토큰은 자동회귀 생성량을 줄인다.
5. 이미지·업로드 서명·대화 기록을 유지한다.
6. 이전 이미지와 새 이미지의 문맥이 혼합되는 것을 방지한다.
7. 아니다. 로컬 컴퓨터의 Ollama 서버 주소이다.


# 23. 실습 과제

## 과제 1. 단일 이미지 멀티턴

이미지 한 장을 업로드하고 5회 이상 후속 질문을 수행한다.

평가 항목:

```text
Object Accuracy
Attribute Accuracy
Context Consistency
Reference Resolution
Hallucination
```

## 과제 2. 이미지 3장 비교

서로 다른 이미지 3장을 업로드한다.

다음 질문을 수행한다.

```text
각 이미지의 객체는?
Picture 1과 Picture 2의 공통점은?
Picture 2와 Picture 3의 차이점은?
Picture 1에만 있는 특징은?
세 이미지를 표 형식으로 비교해줘.
```

## 과제 3. 오류 분석

최소 3개 오류 사례를 저장한다.

```text
질문
모델 답변
실제 이미지 내용
오류 유형
오류 원인
개선 Prompt
```

## 과제 4. UI 확장

다음 중 두 가지 이상을 추가한다.

```text
대화 다운로드
답변 복사
이미지별 이름 지정
System Prompt 입력
Temperature 설정
대화 평가 점수
오류 유형 선택
JSON 결과 저장
```

# 24. 전체 과정 연결

```text
Multimodal Concept
        ↓
Embedding
        ↓
Shared Embedding Space
        ↓
Contrastive Learning
        ↓
CLIP
├── Zero-shot Classification
└── Image Retrieval
        ↓
BLIP
├── Captioning
└── VQA
        ↓
CLIP + BLIP Project
        ↓
Qwen2.5-VL
├── Korean VQA
├── Multi-Image
└── Multi-turn
        ↓
Streamlit Application
├── Image Upload
├── Korean Chat
├── Session State
├── Multi-Image Comparison
└── Context-aware Conversation
```

이번 단계부터 모델 실습이 실제 사용자가 조작할 수 있는 애플리케이션으로 확장된다.

# 25. 다음 단계

다음 단계에서는 현재 Streamlit 앱을 더 실제 서비스에 가깝게 확장할 수 있다.

```text
Streamlit Multimodal Chat
        ↓
대화 결과 저장
        ↓
JSON / CSV Export
        ↓
평가 Dashboard
        ↓
질문별 점수
        ↓
오류 유형 통계
        ↓
VLM 성능 평가
```

다음 Notebook은 **`13_VLM_성능평가_대시보드.ipynb`**로 확장할 수 있다.